# Intent Classification

Decides what kind of message the user sent, so the chatbot knows where to send it (issue #4).

| Intent | Example |
|---|---|
| transaction | اشتريت لاب من مول البستان ب 5000 جنيه |
| spending_query | صرفت كام على الأكل الشهر ده؟ |
| forecast | هصرف كام الشهر الجاي؟ |
| market | سعر الدولار كام؟ |
| advice | احوش ازاي من مرتبي؟ |
| manage | امسح آخر حاجة |
| greeting | ازيك |
| out_of_scope | عايز وصفة كشري |

Run all the cells once, then go to **Try it** at the bottom and type your own message.

## 1. Load the data

In [1]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("../../data/intent_dataset.csv")
print("Dataset shape:", df.shape)
df["intent"].value_counts()

Dataset shape: (2008, 4)


intent
transaction       560
spending_query    279
advice            224
market            223
manage            211
out_of_scope      200
greeting          158
forecast          153
Name: count, dtype: int64

## 2. Clean the text

Arabic digits become normal digits, and different spellings of the same letter become one (أ إ آ → ا, ة → ه, ى → ي).

In [2]:
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")

def normalize(text):
    text = str(text).translate(AR_DIGITS)
    text = re.sub("[إأآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    text = re.sub(r"[\u064B-\u0652\u0640]", "", text)  # tashkeel + tatweel
    return text.lower().strip()

normalize("دفعت ١٢٠٠ جنيه للعربيّة")

'دفعت 1200 جنيه للعربيه'

## 3. Train the model

Character n-grams (2 to 5 letters) + Logistic Regression. Character n-grams work well with Egyptian spelling, typos and Franco-Arabic.

In [3]:
df["clean"] = df["text"].apply(normalize)

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["intent"], test_size=0.2, stratify=df["intent"], random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
    ("classifier", LogisticRegression(max_iter=3000, C=10, class_weight="balanced")),
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print(classification_report(y_test, y_pred, zero_division=0))

Test Accuracy: 0.8905

                precision    recall  f1-score   support

        advice       0.86      0.93      0.89        45
      forecast       0.84      0.90      0.87        30
      greeting       0.97      0.97      0.97        32
        manage       0.88      0.86      0.87        42
        market       0.88      0.96      0.91        45
  out_of_scope       0.87      0.65      0.74        40
spending_query       0.88      0.82      0.85        56
   transaction       0.91      0.96      0.93       112

      accuracy                           0.89       402
     macro avg       0.89      0.88      0.88       402
  weighted avg       0.89      0.89      0.89       402



## 4. The classify function

- A future-tense verb (هصرف، هحوش، will I) turns a spending question into a forecast. The model alone mixes up "هصرف كام" and "صرفت كام".
- If the model is less than 35% sure, the answer is `unsure`, and the chatbot should ask the user what they meant.

In [4]:
FUTURE_RE = re.compile(
    r"(?:^|\s)(?:ه|هي|هن)(?:صرف|حوش|وفر|دفع|فضل|تبقي|كفي|خلص|قبض|يفضل|يتبقي|يكفي|يخلص)\w*"
    r"|\bwill i\b|\bam i going to\b|\bhow much will\b|\bha(?:sraf|7awesh|dfa3)\b|\bhayfdal\b",
    re.I,
)
UNSURE_BELOW = 0.35

def predict_intent(message):
    clean = normalize(message)
    probs = model.predict_proba([clean])[0]
    ranked = sorted(zip(model.classes_, probs), key=lambda x: -x[1])

    intent, confidence = ranked[0]
    note = ""
    if intent == "spending_query" and FUTURE_RE.search(clean):
        intent, note = "forecast", "changed from spending_query by the future-tense rule"
    if confidence < UNSURE_BELOW:
        intent, note = "unsure", f"model's best guess was {ranked[0][0]}"
    return intent, confidence, ranked, note


def classify(message):
    intent, confidence, ranked, note = predict_intent(message)
    print(f"Message:    {message}")
    print(f"Intent:     {intent}" + (f"  ({note})" if note else ""))
    print(f"Confidence: {confidence:.0%}")
    print("Top 3:     ", ", ".join(f"{name} {p:.0%}" for name, p in ranked[:3]))

## 5. Try it

Change the message and run the cell.

In [5]:
message = "سعر الدولار كام"
classify(message)

Message:    سعر الدولار كام
Intent:     market
Confidence: 97%
Top 3:      market 97%, spending_query 2%, out_of_scope 0%


In [6]:
message = "اشتريت لاب من مول البستان ب 5000 جنيه"
classify(message)

Message:    اشتريت لاب من مول البستان ب 5000 جنيه
Intent:     transaction
Confidence: 96%
Top 3:      transaction 96%, advice 1%, manage 1%


In [7]:
message = "هصرف كام الشهر ده؟"
classify(message)

Message:    هصرف كام الشهر ده؟
Intent:     forecast  (changed from spending_query by the future-tense rule)
Confidence: 90%
Top 3:      spending_query 90%, forecast 9%, market 0%


### Many messages at once

In [8]:
messages = [
    "سعر الدولار كام",
    "قبضت المرتب ١٥٠٠٠",
    "صرفت كام على الأكل الشهر ده؟",
    "How much did I spend on Uber this week?",
    "لو فضلت كده هحوش كام آخر السنة",
    "سهم CIB عامل ايه؟",
    "اقدر اشتري عربية بمصاريفي دي؟",
    "امسح آخر حاجة",
    "صباح الخير",
    "مين هيكسب الماتش النهارده؟",
]

rows = []
for m in messages:
    intent, confidence, ranked, note = predict_intent(m)
    rows.append({"message": m, "intent": intent, "confidence": f"{confidence:.0%}", "note": note})

pd.DataFrame(rows)

,message,intent,confidence,note
0,سعر الدولار كام,market,97%,
1,قبضت المرتب ١٥٠٠٠,transaction,47%,
2,صرفت كام على الأكل الشهر ده؟,spending_query,99%,
3,How much did I spend on Uber this week?,spending_query,96%,
4,لو فضلت كده هحوش كام آخر السنة,forecast,89%,
5,سهم CIB عامل ايه؟,market,94%,
6,اقدر اشتري عربية بمصاريفي دي؟,advice,74%,
7,امسح آخر حاجة,manage,96%,
8,صباح الخير,greeting,97%,
9,مين هيكسب الماتش النهارده؟,greeting,43%,


Not perfect yet: off-topic questions like "مين هيكسب الماتش" are the weakest part (they need more training examples). When the model is not sure, the answer is `unsure` instead of a wrong guess.

### Type a message yourself

Uncomment the line below and run the cell. A box appears where you can type your message.

In [9]:
# classify(input("Message: "))